# DA3408 Assignment 3 - NYC Taxi Data Pipeline (PySpark)
Data: [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), yellow taxi Jan 2022 - Jun 2025 (~2 GB).

## 0. Download the data from the TLC page

In [1]:
import os, re, glob, urllib.request

page = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"
headers = {"User-Agent": "Mozilla/5.0"}
os.makedirs("data/raw", exist_ok=True)

html = urllib.request.urlopen(urllib.request.Request(page, headers=headers)).read().decode()
links = sorted(set(re.findall(r'https://[^"\s]+yellow_tripdata_\d{4}-\d{2}\.parquet', html)))
links = [l for l in links if "2022-01" <= re.search(r"\d{4}-\d{2}", l).group() <= "2025-06"]
links.append("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")

for url in links:
    name = url.split("/")[-1]
    path = "data/" + name if name.endswith(".csv") else "data/raw/" + name
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    print(name, round(os.path.getsize(path) / 1e6, 1), "MB")

yellow_tripdata_2022-01.parquet 38.1 MB
yellow_tripdata_2022-02.parquet 45.6 MB
yellow_tripdata_2022-03.parquet 55.7 MB
yellow_tripdata_2022-04.parquet 55.2 MB
yellow_tripdata_2022-05.parquet 55.6 MB
yellow_tripdata_2022-06.parquet 55.4 MB
yellow_tripdata_2022-07.parquet 49.4 MB
yellow_tripdata_2022-08.parquet 49.7 MB
yellow_tripdata_2022-09.parquet 49.6 MB
yellow_tripdata_2022-10.parquet 57.1 MB
yellow_tripdata_2022-11.parquet 50.1 MB
yellow_tripdata_2022-12.parquet 53.6 MB
yellow_tripdata_2023-01.parquet 47.7 MB
yellow_tripdata_2023-02.parquet 47.7 MB
yellow_tripdata_2023-03.parquet 56.1 MB
yellow_tripdata_2023-04.parquet 54.2 MB
yellow_tripdata_2023-05.parquet 58.7 MB
yellow_tripdata_2023-06.parquet 55.0 MB
yellow_tripdata_2023-07.parquet 48.4 MB
yellow_tripdata_2023-08.parquet 48.2 MB
yellow_tripdata_2023-09.parquet 47.9 MB
yellow_tripdata_2023-10.parquet 59.0 MB
yellow_tripdata_2023-11.parquet 56.1 MB
yellow_tripdata_2023-12.parquet 56.8 MB
yellow_tripdata_2024-01.parquet 50.0 MB


## 1. Ingestion

In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType

spark = (SparkSession.builder.master("local[*]").appName("taxi")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())

cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
        "PULocationID", "DOLocationID", "fare_amount", "tip_amount", "total_amount"]

# the files have slightly different column types across years, so cast each one before combining
types = {"passenger_count": "int", "trip_distance": "double", "PULocationID": "int", "DOLocationID": "int",
         "fare_amount": "double", "tip_amount": "double", "total_amount": "double",
         "tpep_pickup_datetime": "timestamp", "tpep_dropoff_datetime": "timestamp"}

df = None
for f in sorted(glob.glob("data/raw/*.parquet")):
    part = spark.read.parquet(f).select([F.col(c).cast(types[c]).alias(c) for c in cols])
    df = part if df is None else df.unionByName(part)

print("rows loaded:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/02 13:02:15 WARN Utils: Your hostname, vmi2701799, resolves to a loopback address: 127.0.1.1; using 82.180.145.105 instead (on interface eth0)
26/10/02 13:02:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/02 13:02:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


rows loaded: 143219428


## 2. Cleansing

In [3]:
before = df.count()

df = df.dropna(subset=cols)          # remove nulls
df = df.dropDuplicates()             # remove duplicates

# keep only trips inside the dataset period and with sensible values
df = df.filter(F.col("tpep_pickup_datetime").between("2022-01-01", "2025-07-01"))
df = df.filter((F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")) &
               (F.col("trip_distance") > 0) & (F.col("fare_amount") > 0))

# format timestamps
df = (df.withColumn("pickup_time", F.date_format("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss"))
        .withColumn("dropoff_time", F.date_format("tpep_dropoff_datetime", "yyyy-MM-dd HH:mm:ss")))

after = df.count()
print("before:", before, "after:", after, "removed:", before - after)

before: 143219428 after: 127591085 removed: 15628343


## 3. Transformation
Heavy join: trips joined with the zone lookup table (once for pickup, once for dropoff). Then a custom Python UDF for average speed, and average speed per hour.

In [4]:
zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True)
zones.show(5)

pickup_zones = zones.select(F.col("LocationID").alias("PULocationID"), F.col("Borough").alias("pickup_borough"), F.col("Zone").alias("pickup_zone"))
dropoff_zones = zones.select(F.col("LocationID").alias("DOLocationID"), F.col("Borough").alias("dropoff_borough"), F.col("Zone").alias("dropoff_zone"))

df = df.join(pickup_zones, "PULocationID", "left").join(dropoff_zones, "DOLocationID", "left")

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [5]:
# custom Python function (UDF): average speed in miles per hour
def avg_speed(distance, seconds):
    if seconds is None or seconds <= 0:
        return None
    return distance / (seconds / 3600)

avg_speed_udf = F.udf(avg_speed, DoubleType())

df = df.withColumn("duration_sec", F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long"))
df = df.withColumn("avg_speed_mph", avg_speed_udf("trip_distance", "duration_sec"))
df = df.filter(F.col("avg_speed_mph").between(1, 80))     # drop impossible speeds
df = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))

# average speed per hour of the day
speed_per_hour = df.groupBy("pickup_hour").agg(F.avg("avg_speed_mph").alias("avg_speed_per_hour"))
speed_per_hour.orderBy("pickup_hour").show(24)

df = df.join(speed_per_hour, "pickup_hour")

26/10/02 13:05:11 WARN DAGScheduler: Broadcasting large task binary with size 1276.1 KiB


26/10/02 13:07:34 WARN DAGScheduler: Broadcasting large task binary with size 1081.7 KiB


26/10/02 13:08:45 WARN DAGScheduler: Broadcasting large task binary with size 1068.6 KiB


+-----------+------------------+
|pickup_hour|avg_speed_per_hour|
+-----------+------------------+
|          0|14.954415599907234|
|          1|14.749131571458143|
|          2|14.714586287701975|
|          3|15.676434572562998|
|          4| 18.77334485021835|
|          5|20.450432622648705|
|          6|16.697081357080933|
|          7|13.269506570758276|
|          8|11.285112392242498|
|          9|10.916356791738808|
|         10|10.618158248245614|
|         11|10.099933398149494|
|         12|10.006442400508874|
|         13|10.047587191179142|
|         14| 9.816867310738592|
|         15| 9.621406215937672|
|         16| 9.707811329987026|
|         17| 9.665679506545809|
|         18|10.136876864686743|
|         19|11.259145867842888|
|         20|12.245106491712821|
|         21|12.809673520858766|
|         22|13.252513664658615|
|         23|14.349113657703274|
+-----------+------------------+



## 4. Export

In [6]:
df.write.mode("overwrite").parquet("output/taxi_features.parquet")

result = spark.read.parquet("output/taxi_features.parquet")
print("rows saved:", result.count())
result.printSchema()
result.show(5)

26/10/02 13:08:48 WARN DAGScheduler: Broadcasting large task binary with size 1312.0 KiB


26/10/02 13:11:27 WARN DAGScheduler: Broadcasting large task binary with size 1133.9 KiB
26/10/02 13:11:27 WARN DAGScheduler: Broadcasting large task binary with size 1139.8 KiB


26/10/02 13:16:27 WARN DAGScheduler: Broadcasting large task binary with size 1125.7 KiB


rows saved: 127324977
root
 |-- pickup_hour: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_time: string (nullable = true)
 |-- dropoff_time: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- duration_sec: long (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- avg_speed_per_hour: double (nullable = true)



+-----------+------------+------------+--------------------+---------------------+---------------+-------------+-----------+----------+------------+-------------------+-------------------+--------------+--------------------+---------------+-----------------+------------+------------------+------------------+
|pickup_hour|DOLocationID|PULocationID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|        pickup_time|       dropoff_time|pickup_borough|         pickup_zone|dropoff_borough|     dropoff_zone|duration_sec|     avg_speed_mph|avg_speed_per_hour|
+-----------+------------+------------+--------------------+---------------------+---------------+-------------+-----------+----------+------------+-------------------+-------------------+--------------+--------------------+---------------+-----------------+------------+------------------+------------------+
|         12|         170|          90| 2022-01-01 12:13:21|  2022-01-